## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:


Please visit Resend at: https://resend.com/

(Resend is a modern email API service for developers.)

Please set up an account - it's free! (at least for the first 3,000 emails per month).

Once you've created an account:

1. **Get your API Key:**
   - Go to API Keys in your dashboard
   - Click "Create API Key"
   - Give it a name (e.g., "Sales Agent")
   - Copy the key to the clipboard

2. **Add the key to your .env file:**
   ```
   RESEND_API_KEY=re_xxxxxxxxxxxxxxxxxxxxxxxxxx
   ```

3. **Set up your domain:**
   - Go to Domains in your dashboard
   - Click "Add Domain"
   - Add your domain (e.g., yourdomain.com)
   - Follow the DNS verification steps
   - Or use the default `onboarding@resend.dev` for testing

4. **Update the sender email:**
   - Make sure to update the `from` field in the functions to use your verified domain
   - Format: `yourname@yourdomain.com` or `onboarding@resend.dev` for testing
   - **Important:** Replace `[youremail@yourdomain.com]` and `[recipient@example.com]` in the code with your actual email addresses


In [ ]:
# !uv pip install resend

In [6]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import resend
import os
import asyncio



In [7]:
load_dotenv(override=True)

True

In [20]:
# Let's just check emails are working for you

def send_test_email(body: str):
    resend.api_key = os.environ.get('RESEND_API_KEY')
    
    params = {
        "from": "onboarding@resend.dev",  # Change to your verified sender domain or use a test address provided by resend
        "to": ["hasnainasif52@gmail.com"],  # Change to your recipient
        "subject": "Sales email",
        "text": body
    }
    
    response = resend.Emails.send(params)
    print(response)
    print(response.get("id"))

send_test_email("Here is my test email body.")


{'id': '05faa388-0484-4b61-a45a-a8f4b8fc1d79'}
05faa388-0484-4b61-a45a-a8f4b8fc1d79


## Step 1: Agent workflow

In [21]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [22]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4.1-nano"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4.1-nano"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4.1-nano"
)

In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email") # run the agent1 with streaming response
# Its adding trace internally with Workflow name as 'Agent workflow' in https://platform.openai.com/logs?api=traces


In [41]:
result

RunResultStreaming(input='Write a cold sales email', new_items=[], raw_responses=[], final_output=None, input_guardrail_results=[], output_guardrail_results=[], context_wrapper=RunContextWrapper(context=None, usage=Usage(requests=0, input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0)), current_agent=Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model='gpt-4.1-nano', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query

In [45]:
result.final_output

'Subject: Simplify Your SOC2 Compliance with ComplAI\n\nHi [Recipient’s Name],\n\nI hope this message finds you well. Ensuring SOC2 compliance can be a complex and resource-intensive process, but it doesn’t have to be. \n\nAt ComplAI, we’ve developed an AI-powered SaaS platform that streamlines compliance management and audit readiness, saving your team time and reducing risk. Our solution provides real-time monitoring, document automation, and audit preparation tools tailored specifically for organizations like yours.\n\nWould you be open to a quick conversation to explore how ComplAI can support your compliance efforts?\n\nLooking forward to connecting.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \nComplAI  \n[Your Contact Information]  \n[Website URL]'

In [43]:
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC2 Compliance with ComplAI

Hi [Recipient’s Name],

I hope this message finds you well. Ensuring SOC2 compliance can be a complex and resource-intensive process, but it doesn’t have to be. 

At ComplAI, we’ve developed an AI-powered SaaS platform that streamlines compliance management and audit readiness, saving your team time and reducing risk. Our solution provides real-time monitoring, document automation, and audit preparation tools tailored specifically for organizations like yours.

Would you be open to a quick conversation to explore how ComplAI can support your compliance efforts?

Looking forward to connecting.

Best regards,  
[Your Name]  
[Your Position]  
ComplAI  
[Your Contact Information]  
[Website URL]

In [ ]:
result_sync = await Runner.run(sales_agent1, input="Write a simple cold sales email") # run the agent1
# Its adding trace internally with Workflow name as 'Agent workflow' in https://platform.openai.com/logs?api=traces
result_sync

RunResult(input='Write a simple cold sales email', new_items=[MessageOutputItem(agent=Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model='gpt-4.1-nano', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_0f01b5d869877a560068f73db4f4ec819795d9db68501cb8fa', conte

In [44]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    ) # its similar to promise.all in javascript

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Simplify Your SOC2 Compliance and Audit Preparation

Hi [Recipient's Name],

I hope this message finds you well. As organizations like yours navigate the complexities of SOC2 compliance, the process can often be time-consuming and resource-intensive.

At ComplAI, we’ve developed an AI-powered SaaS platform designed to streamline your compliance efforts, automate documentation, and prepare you for audits with confidence—reducing risk and saving valuable time.

Would you be open to a brief conversation to explore how ComplAI can support your compliance journey?

Looking forward to your thoughts.

Best regards,  
[Your Name]  
[Your Title]  
ComplAI  
[Your Contact Information]


Subject: Let’s Make SOC2 Compliance Less of a Catastrophe 🧙‍♂️✨

Hi [First Name],

Ever feel like SOC2 compliance is just a fancy term for “finding the secret sauce” with a magnifying glass? Yeah, us too. That’s why we built ComplAI — your AI-powered sidekick that makes prepping for SOC2 audits feel less

In [49]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4.1-nano"
)

In [52]:
"Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(['aaaaaaaaaa', 'bbbbbbbbbb', 'cccccccccc'])

'Cold sales emails:\n\naaaaaaaaaa\n\nEmail:\n\nbbbbbbbbbb\n\nEmail:\n\ncccccccccc'

In [ ]:
message = "Write a cold sales email"

with trace("Selection from 3 sales' emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: 🎯 Hit Your SOC2 Goals — No Crystal Ball Required! 


Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [ ]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4.1-nano",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4.1-nano",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4.1-nano",
)

In [56]:
sales_agent1

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model='gpt-4.1-nano', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [57]:
# Put the decorator above the function, that we want to use as a tool

@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    resend.api_key = os.environ.get('RESEND_API_KEY')

    params = {
        "from": "onboarding@resend.dev",  # Change to your verified sender domain or use a test address provided by resend
        "to": ["hasnainasif52@gmail.com"],  # Change to your recipient
        "subject": "Sales email",
        "text": body
    }
    
    response = resend.Emails.send(params)
    return {"status": "success", "id": response.get("id")}

### This has automatically been converted into a tool, with the boilerplate json created

In [58]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f734aa45e40>, strict_json_schema=True, is_enabled=True)

### And you can also convert an Agent into a tool

In [59]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73495074c0>, strict_json_schema=True, is_enabled=True)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [60]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73495065c0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73495058a0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'req

## And now it's time for our Sales Manager - our planning agent

In [ ]:
# instructions ="You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
# You never generate sales emails yourself; you always use the tools. \
# You try all 3 sales_agent tools once before choosing the best one. \
# You pick the single best email and use the send_email tool to send the best email (and only the best email) to the user."

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool only once to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(name="Efficient Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

# with trace("Sales manager, Selecting one best email and sending it out"):
with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

print(result)

RunResult:
- Last agent: Agent(name="Efficient Sales Manager", ...)
- Final output (str):
    I've sent out the two best drafted emails to the respective recipients. If you need further assistance, feel free to ask!
- 11 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [67]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4.1-nano")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4.1-nano")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [68]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    resend.api_key = os.environ.get('RESEND_API_KEY')
    
    params = {
        "from": "onboarding@resend.dev",  # Change to your verified sender domain or use a test address provided by resend
        "to": ["hasnainasif52@gmail.com"],  # Change to your recipient
        "subject": subject,
        "html": html_body
    }
    
    response = resend.Emails.send(params)
    return {"status": "success", "id": response.get("id")}

In [76]:
tools = [subject_tool, html_tool, send_html_email]

In [77]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73496a79c0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73496a5440>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', description='Send out an email with the given subject and HTML body to all sale

In [78]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")


### Now we have 3 tools and 1 handoff

In [79]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73495065c0>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f73495058a0>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'requi

In [ ]:
# sales_manager_instructions = "You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
# You never generate sales emails yourself; you always use the tools. \
# You try all 3 sales agent tools at least once before choosing the best one. \
# You can use the tools multiple times if you're not satisfied with the results from the first try. \
# You select the single best email using your own judgement of which email will be most effective. \
# After picking the email, you handoff to the Email Manager agent to format and send the email."
sales_manager_instructions = "You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
You never generate sales emails yourself; you always use the tools. \
You try all 3 sales agent tools at least once before choosing the best one. \
You select the single best email using your own judgement of which email will be most effective. \
After picking the email, you handoff to the Email Manager agent to format and send the email."


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has announced their Agent Development Kit (ADK) which is in early preview. It's still under development, so it's too early for us to cover it here. But it's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!